In [ ]:
#quick inspection of the .edf file (manually downloaded in data, for example):

import pyedflib

file_path = r"C:\Users\name\data\file.edf"  #example path, change as needed

with pyedflib.EdfReader(file_path) as f:
    print("Number of signals:", f.signals_in_file)
    print("Sampling frequencies:", [f.getSampleFrequency(i) for i in range(f.signals_in_file)])
    print("Channel names:", f.getSignalLabels())
    print("File duration (s):", f.getFileDuration())
    print("Number of samples per channel:", f.getNSamples())

#in agreement with .pos file

Number of signals: 93
Sampling frequencies: [1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0]
Channel names: ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'FZ', 'CZ', 'PZ', 'SO1', 'SO2', 'F9', 'F10', 'ZY1', 'ZY2', 'T9', 'T10', 'P9', 'P10', 'AF7', 'AF3'

In [ ]:
#quick channel check:

import pyedflib

with pyedflib.EdfReader(r"C:\Users\name\data\file.edf") as f: #example path, change as needed
    labels = f.getSignalLabels()
    
    eeg = [ch for ch in labels if ch not in ['ChEMG1','ChEMG2','RLEG-','RLEG+','LLEG-','LLEG+','ECG1','ECG2','EOG1','EOG2']]
    aux  = [ch for ch in labels if ch in ['ChEMG1','ChEMG2','RLEG-','RLEG+','LLEG-','LLEG+','ECG1','ECG2','EOG1','EOG2']]
    
    print(f"EEG channels: {len(eeg)}")
    print(f"Auxiliary channels: {len(aux)}")
    print("Aux channels:", aux)  

EEG channels: 83
Auxiliary channels: 10
Aux channels: ['ChEMG1', 'ChEMG2', 'RLEG-', 'RLEG+', 'LLEG-', 'LLEG+', 'ECG1', 'EOG1', 'ECG2', 'EOG2']


In [ ]:
# ============================
# main.ipynb - Controlled Download + Processing 
# ============================

from pathlib import Path
import gc

try:
    from tqdm.auto import tqdm
except ImportError:
    # fallback no-op shim so a missing tqdm install doesn't crash the run
    def tqdm(iterable, *args, **kwargs):
        return iterable
    tqdm.write = print

from download_data import (
    download_shared_files,
    download_subject,
    get_subject_edf_path,
    get_subject_scoring_path,
    get_subject_artifact_path,
    get_available_subjects,
)
from preprocessing import process_subject

DATA_ROOT = Path("./anphy_sleep_data")
DATA_ROOT.mkdir(exist_ok=True)

OUTPUT_DIR = Path("./graphs_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- quick-test switch ---
# True  -> process just subject 1, as a smoke test of the whole pipeline
# False -> process every available subject on the dataset

QUICK_TEST = True 

subjects_to_process = [18, 19, 22, 23, 24, 25] if QUICK_TEST else get_available_subjects()
print(f"QUICK_TEST={QUICK_TEST} -> {len(subjects_to_process)} subject(s) queued: {subjects_to_process}")

# 1. Download shared files (only once)
print("\nDownloading shared files (artifact matrix + .pos)...")
download_shared_files(DATA_ROOT)

# Delete the artifact_matrix.zip after extraction
artifact_zip = DATA_ROOT / "artifact_matrix.zip"
if artifact_zip.exists():
    try:
        artifact_zip.unlink()
        print("Deleted artifact_matrix.zip to save space.")
    except Exception as e:
        print(f"Could not delete artifact_matrix.zip: {e}")

QUICK_TEST=True -> 6 subject(s) queued: [18, 19, 22, 23, 24, 25]

  details_information_for_healthy_subjects.csv already exists.
  co-registered_average_positions.pos already exists.
  artifact_matrix.zip: 100.0%
Deleted artifact_matrix.zip to save space.


In [ ]:
import preprocessing
import connectivity
import importlib
importlib.reload(preprocessing)          # force reload the updated .py file
importlib.reload(connectivity)

from preprocessing import process_subject
from preprocessing import get_emg_channel_indices

print("process_subject is available:", callable(process_subject))
print("process_subject is available:", callable(get_emg_channel_indices))

print("MAX_BAD_FRACTION in the loaded module:", preprocessing.MAX_BAD_FRACTION)

#import inspect


process_subject is available: True
process_subject is available: True
MAX_BAD_FRACTION in the loaded module: 0.15


In [21]:
# 2. Download + process each subject, one at a time

results = []   # list of (subject_id, h5_path) for successes
failed = []    # list of (subject_num, reason) for skipped/failed subjects

for subj_num in subjects_to_process:
    subj_id = f"EPCTL{subj_num:02d}"
    print(f"\n{'='*70}\nPROCESSING SUBJECT {subj_id}\n{'='*70}")

    # === DOWNLOAD ===
    subj_dir = download_subject(subj_num, DATA_ROOT, delete_zip=True)
    if not subj_dir:
        print("  Skipped (subject unavailable or download failed).")
        failed.append((subj_num, "download_failed"))
        continue

    # === FIND THE MAIN EDF FILE ===
    main_edf = get_subject_edf_path(subj_num, DATA_ROOT)
    if not main_edf or not main_edf.exists():
        print("  No .edf file found! Skipping.")
        failed.append((subj_num, "no_edf"))
        continue

    # === FIND SCORING + ARTIFACT FILES ===
    scoring_path = get_subject_scoring_path(subj_num, DATA_ROOT)
    if not scoring_path or not scoring_path.exists():
        print("  No scoring .txt file found! Skipping.")
        failed.append((subj_num, "no_scoring"))
        continue

    artifact_path = get_subject_artifact_path(subj_num, DATA_ROOT)
    if not artifact_path:
        print("  No artifact matrix found -- proceeding WITHOUT artifact rejection.")

    print(f"  EDF file:     {main_edf.name}")
    print(f"  Scoring file: {scoring_path.name}")
    print(f"  Artifact:     {artifact_path.name if artifact_path else 'none'}")

    # === PROCESS: stream epochs, build per-band wPLI graphs, write .h5 ===
    try:
        h5_path = process_subject(
            subject_id=subj_id,
            edf_path=main_edf,
            scoring_path=scoring_path,
            artifact_path=artifact_path,
            pos_path=DATA_ROOT / "co-registered_average_positions.pos",   # added: bad channel detection+interpolation
            output_dir=OUTPUT_DIR,
            
        )
        print(f"  Saved graphs to: {h5_path}")
        results.append((subj_id, h5_path))
    except Exception as e:
        print(f"  FAILED processing {subj_id}: {e!r}")
        failed.append((subj_num, f"process_subject_error: {e}"))
        continue

    # === CLEAN UP raw EDF to save disk space ===
    deleted_count = 0
    for edf in subj_dir.glob("**/*.edf"):
        try:
            edf.unlink()
            deleted_count += 1
        except Exception as e:
            print(f"  Could not delete {edf.name}: {e}")
    print(f"  Deleted {deleted_count} raw EDF file(s)")

    gc.collect()

print(f"\nAll requested subjects processed.")
print(f"  Succeeded: {len(results)}  -> {[s for s, _ in results]}")
print(f"  Failed/skipped: {len(failed)}  -> {failed}")


PROCESSING SUBJECT EPCTL18
  EPCTL18.zip: 100.0%
Extracting EPCTL18...
  EDF file:     EPCTL18.edf
  Scoring file: EPCTL18.txt
  Artifact:     EPCTL18_artndxn.mat


EPCTL18 epochs: 0epoch [00:00, ?epoch/s]

  Saved graphs to: graphs_output\EPCTL18.h5
  Deleted 1 raw EDF file(s)

PROCESSING SUBJECT EPCTL19
  EPCTL19.zip: 100.0%
Extracting EPCTL19...
  EDF file:     EPCTL19.edf
  Scoring file: EPCTL19.txt
  Artifact:     EPCTL19_artndxn.mat


EPCTL19 epochs: 0epoch [00:00, ?epoch/s]

  Saved graphs to: graphs_output\EPCTL19.h5
  Deleted 1 raw EDF file(s)

PROCESSING SUBJECT EPCTL22
  EPCTL22.zip: 100.0%
Extracting EPCTL22...
  EDF file:     EPCTL22.edf
  Scoring file: EPCTL22.txt
  Artifact:     EPCTL22_artndxn.mat


EPCTL22 epochs: 0epoch [00:00, ?epoch/s]

  Saved graphs to: graphs_output\EPCTL22.h5
  Deleted 1 raw EDF file(s)

PROCESSING SUBJECT EPCTL23
  EPCTL23.zip: 100.0%
Extracting EPCTL23...
  EDF file:     EPCTL23.edf
  Scoring file: EPCTL23.txt
  Artifact:     EPCTL23_artndxn.mat


EPCTL23 epochs: 0epoch [00:00, ?epoch/s]

KeyboardInterrupt: 

In [ ]:
#INITIAL SANITY CHECK: subject, recorded bands, channel count, adjacency matrix shape, wPLI values, epochs and artifact matrix.

from sanity_check import sanity_check

sanity_check("graphs_output\EPCTL01.h5") #example

--- EPCTL01.h5 ---
subject_id: EPCTL01
bands recorded in attrs: ['delta', 'theta', 'alpha', 'sigma', 'beta']
[OK] all expected datasets present
channel_names: 83 channels
[OK] channel count matches expected (83)
epoch_idx: 952 entries, stage: 952 entries
delta: shape (952, 83, 83)
theta: shape (952, 83, 83)
alpha: shape (952, 83, 83)
sigma: shape (952, 83, 83)
beta: shape (952, 83, 83)
[OK] all bands have identical, consistent shapes
delta: sampled range [0.0000, 0.5752], NaNs=0, <0=0, >1=0, nonzero=98.8%
theta: sampled range [0.0000, 0.5567], NaNs=0, <0=0, >1=0, nonzero=98.8%
alpha: sampled range [0.0000, 0.4974], NaNs=0, <0=0, >1=0, nonzero=98.8%
sigma: sampled range [0.0000, 0.6431], NaNs=0, <0=0, >1=0, nonzero=98.8%
beta: sampled range [0.0000, 0.6560], NaNs=0, <0=0, >1=0, nonzero=98.8%
Stage breakdown:
  N1            74 epochs
  N2           471 epochs
  N3           185 epochs
  REM          176 epochs
  Wake          46 epochs
epoch_idx range: 0-951 (span 952, 952 stored, 0 dro

True

In [1]:
# Quality-control check: assess whether bad-channel interpolation may have
# affected the strongest delta-band wPLI values for one processed subject.
#
# This reads, but does not modify:
# - the original subject-specific artifact matrix (.mat), which indicates
#   which channels were flagged bad in each original epoch; and
# - the processed connectivity file (.h5), which contains the saved wPLI
#   graphs, sleep-stage labels, original epoch indices, and channel names.
#
# The three checks below:
# 1. Locate the largest delta-band wPLI value and determine whether either
#    channel in that edge had been flagged bad/interpolated in that epoch.
# 2. Compare all delta-band edges involving an interpolated channel against
#    edges between two originally clean channels, to distinguish an isolated
#    high value from a systematic interpolation-related difference.
# 3. Perform a rough channel-order plausibility check, because the artifact
#    matrix has no channel-name metadata and is assumed to use the same
#    channel order as the processed HDF5 graphs.
#
# This is a diagnostic/sensitivity check only: it does not reprocess data,
# alter the EDF recordings, or overwrite the existing .h5 file.
from download_data import get_subject_artifact_path
from preprocessing import read_subject_graphs
from pathlib import Path
DATA_ROOT = Path("./anphy_sleep_data")
OUTPUT_DIR = Path("./graphs_output")
from check_max_wpli_source import (
    find_max_wpli_source,
    compare_interpolated_vs_clean_edges,
    channel_bad_fraction_plausibility_check,
)

artifact_path = get_subject_artifact_path(24, DATA_ROOT)
h5_path = OUTPUT_DIR / "EPCTL24.h5"

# 1. What's the maximum wPLI value?
find_max_wpli_source(h5_path, artifact_path, band="delta")

# 2. the systematic version -- is this a one-off, or a pattern?
compare_interpolated_vs_clean_edges(h5_path, artifact_path, band="delta")

# 3. the deeper question this raised -- does the channel order even line up?
result = read_subject_graphs(h5_path)  # or however you already have channel_names loaded
channel_bad_fraction_plausibility_check(artifact_path, list(result["channel_names"]))

Max delta wPLI = 0.9979
  stored epoch position: 652 (original scoring-file epoch index: 665)
  channel pair: CZ <-> FC1
  CZ flagged bad in this epoch? True
  FC1 flagged bad in this epoch? False
  [FLAG] one of the channels in the max-value pair was interpolated in this epoch -- this specific value may be an interpolation artifact rather than a genuine synchronization finding.
--- delta band: interpolated-touching edges vs. clean edges ---
n edges touching an interpolated channel: 58779
  mean=0.2312  median=0.1945  max=0.9979
n clean edges (sampled): 84678
  mean=0.2454  median=0.2089  max=0.9969
mean difference (interpolated-touching minus clean): -0.0142
[OK] no large systematic difference between the two groups.
Channels ranked by overall bad-fraction (worst first):
  CZ       75.4%
  F10      5.6%
  T3       4.6%
  F11      4.6%
  CP4      4.6%
  SO2      4.5%
  P10      4.5%
  TP11     4.5%
  CP2      4.5%
  Fp1      4.4%
  Fp2      4.4%
  F3       4.4%
  F4       4.4%
  C3    

**Subject 06:**

**Check 1 — `find_max_wpli_source`**
Looks at the single highest wPLI value in the file and checks if either channel involved was interpolated. Result: the max value (0.9452) came from two clean, non-interpolated channels — a real measurement, not an artifact. (But only proves one example, not a general pattern.)

**Check 2 — `compare_interpolated_vs_clean_edges`**
Compares average wPLI across *all* edges that touch an interpolated channel vs. all fully-clean edges. This is the real test for systematic bias. Result: nearly identical averages (0.2125 vs 0.2185) across tens of thousands of edges — interpolation isn't inflating or distorting connectivity values overall.

**Check 3 — `channel_bad_fraction_plausibility_check`**
Checks a separate, more basic assumption: whether the artifact-matrix columns actually line up with the right electrodes. Ranks channels by how often they're flagged bad — posterior (back-of-head) electrodes are expected to top the list, since they typically have worse contact in real recordings. Result: yes, posterior channels (`P6, P9, T5, PZ, OZ, O1, O2`, etc.) dominate the "worst" list — consistent with correct channel alignment, not proof, but a good sign.

In [ ]:
# Follow-up quality-control check: investigate whether the apparent
# interpolation-related wPLI effect is concentrated in subjects with more
# bad channels per epoch, and whether it is strongest when BOTH endpoints
# of a connectivity edge were interpolated.
#
# For each selected subject, this reads:
# - the original artifact matrix (.mat), which records which EEG channels
#   were marked bad in each epoch; and
# - the already processed connectivity graph file (.h5), which contains
#   the saved per-epoch wPLI matrices.
#
# The two checks below:
# 1. Summarize how many channels were flagged bad per epoch. This helps
#    determine whether a flagged subject has unusually frequent or severe
#    bad-channel episodes, making interpolation effects more plausible.
# 2. Split delta-band wPLI edges into three categories:
#      - both_clean: neither endpoint was interpolated
#      - one_interpolated: exactly one endpoint was interpolated
#      - both_interpolated: both endpoints were interpolated
#
# The key question is whether elevated wPLI is concentrated specifically
# in both-interpolated edges. If so, it would be consistent with a possible
# interpolation artifact: two reconstructed channels may share information
# from overlapping nearby reference channels and therefore appear more
# phase-synchronized than they truly were.
#
# Add several unflagged/comparison subjects as well, so that any pattern
# can be judged against ordinary recordings rather than only the subjects
# that initially looked unusual.
#
# This is a read-only diagnostic check. It does not modify the original
# artifact matrices, reprocess EDF files, or overwrite any .h5 outputs.

from pathlib import Path
from check_max_wpli_source import (
    per_epoch_bad_channel_summary,
    compare_interpolated_edges_three_way,
)
from download_data import get_subject_artifact_path

DATA_ROOT = Path("./anphy_sleep_data")
OUTPUT_DIR = Path("./graphs_output")

subjects_to_check = [27, 28]  # flagged; add e.g. 6, 9, 1 for comparison against clean subjects

for subj in subjects_to_check:
    artifact_path = get_subject_artifact_path(subj, DATA_ROOT)
    h5_path = OUTPUT_DIR / f"EPCTL{subj:02d}.h5"

    per_epoch_bad_channel_summary(artifact_path, subject_label=f"subj {subj}")
    compare_interpolated_edges_three_way(h5_path, artifact_path, band="delta")

--- per-epoch bad-channel counts (subj 27) ---
n_epochs: 913
mean bad channels/epoch: 1.69
median: 0.0  max: 83
epochs with >= 1 bad channels: 6.5%
epochs with >= 3 bad channels: 2.1%
epochs with >= 5 bad channels: 2.0%
epochs with >= 10 bad channels: 2.0%
epochs with >= 12 bad channels: 2.0%
epochs with >= 17 bad channels: 2.0%
--- delta band: three-way edge comparison ---
both_clean         n= 82246  mean=0.2482  median=0.2107  max=0.9925
one_interpolated   n=  3840  mean=0.3756  median=0.3386  max=0.9950
both_interpolated  n=     7  mean=0.3428  median=0.2558  max=0.7846
mean difference (one_interpolated minus both_clean): +0.1274 [FLAG]
mean difference (both_interpolated minus both_clean): +0.0945 [FLAG]
mean difference (both_interpolated minus one_interpolated): -0.0329
[INFO] inflation is not clearly concentrated in both-interpolated edges specifically -- one_interpolated and both_interpolated look similar, so the overlapping-neighbor hypothesis is not strongly supported by this 

In [ ]:
# Mechanistic follow-up quality-control check: test whether elevated wPLI
# involving an interpolated channel is strongest for physically nearby
# channels on the scalp.
#
# During spherical-spline interpolation, a bad electrode is reconstructed
# from surrounding clean electrodes. Nearby electrodes typically contribute
# more strongly to that reconstruction than distant electrodes. Therefore,
# if interpolation is artificially increasing wPLI, edges between an
# interpolated channel and nearby clean channels may show higher wPLI than
# edges from that same interpolated channel to more distant clean channels.
#
# For each selected subject, this function:
# 1. Reads the processed delta-band wPLI graphs from the subject's .h5 file.
# 2. Uses the original artifact matrix to select edges with exactly one
#    interpolated endpoint in each retained epoch.
# 3. Uses the electrode coordinates in the .pos file to calculate the
#    physical distance between the two electrodes in every selected edge.
# 4. Reports the correlation between electrode distance and wPLI, plus
#    mean wPLI in distance bins from nearest to farthest.
#
# Evidence supportive of interpolation-related inflation would be a negative
# relationship: high wPLI for edges near an interpolated channel that
# decreases as electrode distance increases.
#
# This is a read-only diagnostic check. It does not alter the montage file,
# artifact matrix, raw EDF data, or processed .h5 graph outputs.

from check_max_wpli_source import check_interpolation_distance_effect

for subj in [27, 28]:
    artifact_path = get_subject_artifact_path(subj, DATA_ROOT)
    h5_path = OUTPUT_DIR / f"EPCTL{subj:02d}.h5"
    pos_path = pos_path=DATA_ROOT / "co-registered_average_positions.pos"

    check_interpolation_distance_effect(h5_path, artifact_path, pos_path, band="delta")


--- delta band: interpolation-distance effect (one_interpolated edges only) ---
n edges: 3840
correlation (distance to interpolated channel vs. wPLI): r=-0.066
[SUPPORTIVE] negative correlation -- wPLI tends to be higher for edges closer to the interpolated channel, consistent with spline-weight-driven inflation (nearby channels contribute more to the reconstruction, so they end up looking more synchronized with it).

By distance bin (nearest to farthest):
  bin 1/4  dist [0.009, 0.097)m  n=   950  mean wPLI=0.4178
  bin 2/4  dist [0.097, 0.136)m  n=   963  mean wPLI=0.3554
  bin 3/4  dist [0.136, 0.170)m  n=   966  mean wPLI=0.3525
  bin 4/4  dist [0.170, 0.205)m  n=   961  mean wPLI=0.3776

nearest-bin minus farthest-bin mean wPLI: +0.0403
--- delta band: interpolation-distance effect (one_interpolated edges only) ---
n edges: 2046
correlation (distance to interpolated channel vs. wPLI): r=-0.046
[NOT SUPPORTIVE] no clear negative correlation -- distance to the interpolated channel d

“Bad channels (≤15% per epoch) were interpolated using spherical splines. Spherical-spline interpolation is known to modestly inflate phase-based connectivity estimates involving the reconstructed channels (Kang et al., 2015). We quantified this effect in our data and observed a mean increase of approximately 0.05 in wPLI for edges touching interpolated channels. We report this as a limitation and did not apply post-hoc correction. Reason: maximum three bad channels per epoch.”


---

Bad channels were identified from the provided artifact matrix and interpolated with spherical splines (MNE-Python) whenever the proportion of bad channels in an epoch did not exceed 15%. Spherical-spline interpolation is known to introduce a modest but systematic inflation of phase-based connectivity estimates involving the reconstructed channels (Kang et al., 2015). We quantified this effect in several subjects (EPCTL13, 20, 24, 27, 28 and 29). Across these recordings the local increase in wPLI for edges that touched an interpolated channel ranged from approximately +0.05 to +0.13 relative to edges between two clean channels. Importantly, interpolation was required in only a small minority of epochs (typically a few percent of the night) and almost always involved only one or two channels per epoch; epochs with many simultaneous bad channels were already excluded by the 15% threshold. Because the bias is therefore confined to a limited subset of edges, its impact on global graph metrics remains modest. We retained the interpolated data without post-hoc correction and report this residual inflation as a limitation of the present analyses.

---